# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FarisElbaz/ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding one: Content experiences a significant increase in impressions and clicks in the 30-90 day refresh window compared to an unrefreshed baseline.

Methodology Question: How were counterfactual controls established to isolate the true treatment effect from site-wide momentum and editorial selection bias?

Context:
Content items that are selected by editorial teams arent randomized, they often are of higher historical brand quality, active backlink profiles, or renewed internal promotional push. Although, without matching on pre-treatment velocity, domain authority, and seasonality, attributing the entire post-intervention delta strictly to the refresh action risks conflating underlying domain momentum with content-level optimization.

Finding two:
The paper asserts that early warning indicators—specifically trailing engagement velocity and ranking position instability—consistently predict content decay trajectories across diverse client verticals.

Methodology Question: Does the validation scheme employ strict out-of-domain (grouped by domain/client) cross-validation, and how sensitive is the decay label boundary to differing baseline traffic scales?

Context: A 20% decline on a niche B2B technical guide (e.g., losing 10 clicks) represents standard Poisson noise, whereas a 20% decline on an enterprise eCommerce head term (losing 10,000 clicks) represents severe algorithmic demotion. If the split was performed randomly across all URLs, the model risks memorizing client-specific baseline noise patterns rather than learning domain-agnostic decay signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import json
import os
import duckdb
from google.colab import userdata
from huggingface_hub import login
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, KFold

# Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

# 1. Hugging Face Login & DuckDB Connection
hf_token = userdata.get("hf_copllab_access")
login(token=hf_token)
print("Successfully logged in to Hugging Face Hub!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet"

# 2. Extract Pre-Cutoff Features and Forward Target Window
dataset_sql = f"""
WITH pre_cutoff AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_14d,
        SUM(gsc_clicks) AS clicks_14d,
        AVG(gsc_avg_position) AS avg_pos,
        STDDEV(gsc_avg_position) AS std_pos,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-01' AND '2026-03-07') AS clicks_w1,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-08' AND '2026-03-14') AS clicks_w2,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN '2026-03-01' AND '2026-03-07') AS imp_w1,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN '2026-03-08' AND '2026-03-14') AS imp_w2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
      AND report_date <= '2026-03-14'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
),
post_cutoff AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_forward
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN '2026-03-15' AND '2026-03-28'
    GROUP BY 1, 2
)
SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.imp_14d,
    p.clicks_14d,
    p.avg_pos,
    COALESCE(p.std_pos, 0.0) AS std_pos,
    p.clicks_14d * 1.0 / NULLIF(p.imp_14d, 0) AS observed_ctr,
    p.clicks_w1,
    p.clicks_w2,
    (p.clicks_w2 - p.clicks_w1) * 1.0 / NULLIF(p.clicks_w1, 0) AS click_velocity_delta,
    (p.imp_w2 - p.imp_w1) * 1.0 / NULLIF(p.imp_w1, 0) AS imp_velocity_delta,
    CASE
        WHEN COALESCE(f.clicks_forward, 0) >= (p.clicks_14d * 1.2) AND COALESCE(f.clicks_forward, 0) >= 10 THEN 1
        WHEN COALESCE(f.clicks_forward, 0) >= 25 THEN 1
        ELSE 0
    END AS target_high_opportunity,
    COALESCE(f.clicks_forward, 0) AS actual_forward_clicks
FROM pre_cutoff p
LEFT JOIN post_cutoff f
  ON p.client_hash_id = f.client_hash_id
 AND p.content_hash_id = f.content_hash_id;
"""

print("Executing warehouse extraction...")
df = con.sql(dataset_sql).df()
print(
    f"Data loaded: {len(df)} content items across"
    f" {df['client_hash_id'].nunique()} client domains."
)

features = [
    "imp_14d",
    "clicks_14d",
    "avg_pos",
    "std_pos",
    "observed_ctr",
    "clicks_w1",
    "clicks_w2",
    "click_velocity_delta",
    "imp_velocity_delta",
]

df["click_velocity_delta"] = (
    df["click_velocity_delta"].fillna(0.0).clip(-5.0, 5.0)
)
df["imp_velocity_delta"] = df["imp_velocity_delta"].fillna(0.0).clip(-5.0, 5.0)
df["observed_ctr"] = df["observed_ctr"].fillna(0.0)

X = df[features]
y = df["target_high_opportunity"]
groups = df["client_hash_id"]

# 3. Model Definition
lgb_params = {
    "n_estimators": 150,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "verbose": -1,
}

# --- Design A: Naive Random K-Fold (Client Leakage Permitted) ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_random = np.zeros(len(df))

for tr_idx, va_idx in kf.split(X, y):
  clf = lgb.LGBMClassifier(**lgb_params)
  clf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
  oof_random[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]

# --- Design B: Honest GroupKFold (Grouped strictly by client domain) ---
gkf = GroupKFold(n_splits=5)
oof_grouped = np.zeros(len(df))

for tr_idx, va_idx in gkf.split(X, y, groups=groups):
  clf = lgb.LGBMClassifier(**lgb_params)
  clf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
  oof_grouped[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]

df["pred_prob_random"] = oof_random
df["pred_prob_grouped"] = oof_grouped

# 4. Before / After Comparison Table
split_audit_results = [
    {
        "Validation Split Design": "Naive Random 5-Fold (Before - Leakage Risk)",
        "ROC-AUC": round(roc_auc_score(y, oof_random), 4),
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_random), 4),
        "Brier Score (Calibration)": round(brier_score_loss(y, oof_random), 4),
    },
    {
        "Validation Split Design": "Honest Client-Grouped 5-Fold (After - Audited)",
        "ROC-AUC": round(roc_auc_score(y, oof_grouped), 4),
        "PR-AUC (Avg Precision)": round(
            average_precision_score(y, oof_grouped), 4
        ),
        "Brier Score (Calibration)": round(
            brier_score_loss(y, oof_grouped), 4
        ),
    },
]

audit_table_df = pd.DataFrame(split_audit_results)
print("\n=== SPLIT AUDIT: BEFORE VS. AFTER COMPARISON ===")
print(audit_table_df.to_markdown(index=False))

# 5. Extract Concrete Failure Examples
df["error_magnitude"] = np.abs(df["pred_prob_grouped"] - df["target_high_opportunity"])

# False Positives: Model strongly predicted high opportunity (prob > 0.8), but target was 0
fps = df[(df["target_high_opportunity"] == 0) & (df["pred_prob_grouped"] > 0.75)].sort_values(
    "pred_prob_grouped", ascending=False
).head(3)

# False Negatives: Target was 1, but model predicted near zero (prob < 0.15)
fns = df[(df["target_high_opportunity"] == 1) & (df["pred_prob_grouped"] < 0.20)].sort_values(
    "pred_prob_grouped", ascending=True
).head(3)

print("\n--- Failure Examples: False Positives (High Confidence Over-Predictions) ---")
print(fps[["content_hash_id", "imp_14d", "avg_pos", "observed_ctr", "pred_prob_grouped", "actual_forward_clicks"]].to_markdown(index=False))

print("\n--- Failure Examples: False Negatives (Surprise Growth Under-Predictions) ---")
print(fns[["content_hash_id", "imp_14d", "avg_pos", "observed_ctr", "pred_prob_grouped", "actual_forward_clicks"]].to_markdown(index=False))

# Save audit receipt to work/outputs/
with open("work/outputs/w06_audit_metrics.json", "w") as f:
  json.dump(
      {
          "split_comparison": split_audit_results,
          "fp_sample_ids": fps["content_hash_id"].tolist(),
          "fn_sample_ids": fns["content_hash_id"].tolist(),
      },
      f,
      indent=2,
  )
print("\nAudit receipt saved to work/outputs/w06_audit_metrics.json")

Successfully logged in to Hugging Face Hub!
Executing warehouse extraction...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded: 90593 content items across 40 client domains.

=== SPLIT AUDIT: BEFORE VS. AFTER COMPARISON ===
| Validation Split Design                        |   ROC-AUC |   PR-AUC (Avg Precision) |   Brier Score (Calibration) |
|:-----------------------------------------------|----------:|-------------------------:|----------------------------:|
| Naive Random 5-Fold (Before - Leakage Risk)    |    0.9569 |                   0.7129 |                      0.0337 |
| Honest Client-Grouped 5-Fold (After - Audited) |    0.9175 |                   0.5613 |                      0.0433 |

--- Failure Examples: False Positives (High Confidence Over-Predictions) ---
| content_hash_id          |   imp_14d |   avg_pos |   observed_ctr |   pred_prob_grouped |   actual_forward_clicks |
|:-------------------------|----------:|----------:|---------------:|--------------------:|------------------------:|
| content_c5e2e71596d4023c |      3868 |   4.60427 |     0.0400724  |            0.991555 |      

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Observation cutoff integrity: Every input feature is computed using records with report date <= 2026-03-14, Zero rows from march 14th onwards were accessible during feature calculation.

Target separation:
The label (target_high_opportunity) is calculated using post cut off aggregations strictly between 2026-03-15 to 2026-03-28. No forward traffic variables were accessible.

Exclusion of Intermediate System Proxies: No proprietary client indicators, platform refresh status flags, or manual editorial priority tags were passed into training.

Ablation Insight: Moving from a random split to a client-grouped split isolates the true generalizable performance. In random splits, models inadvertently exploit shared client domain characteristics (e.g., domain-wide click baselines across pages belonging to the same client), producing artificially optimistic PR-AUC metrics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Overreaching Claim 1: "The model accurately identifies which content pieces will capture massive traffic growth after an update."$\rightarrow$ Public-Safe Revision: "Under client-grouped validation, the gradient-boosted classifier demonstrated a measured PR-AUC of 0.561, providing directional decision-support to prioritize URLs with favorable historical impression velocity."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.